# Data Wrangling

Loads the launch table, checks missing values, counts launches by site / orbit / outcome, builds the binary **Class** label (1 = first stage landed) and exports `data/spacex_wrangled.csv`.

In [1]:
import pandas as pd, numpy as np
df=pd.read_csv("data/spacex_launch_table.csv"); df.columns=[c.strip() for c in df.columns]
for c in ["Launch_Site","Orbit","Booster_Version","Landing_Outcome","Mission_Outcome"]: df[c]=df[c].str.strip()
df.shape

(101, 10)

In [2]:
df.isnull().sum()

Date                 0
Time (UTC)           0
Booster_Version      0
Launch_Site          0
Payload              0
PAYLOAD_MASS__KG_    0
Orbit                0
Customer             0
Mission_Outcome      0
Landing_Outcome      0
dtype: int64

In [3]:
print(df.Launch_Site.value_counts(),"\n")
print(df.Orbit.value_counts(),"\n")
df.Landing_Outcome.value_counts()

Launch_Site
CCAFS SLC-40    34
CCAFS LC-40     26
KSC LC-39A      25
VAFB SLC-4E     16
Name: count, dtype: int64 

Orbit
GTO            30
LEO (ISS)      26
LEO            25
Polar LEO       8
SSO             6
MEO             3
HEO             2
Sub-orbital     1
Name: count, dtype: int64 



Landing_Outcome
Success                   38
No attempt                22
Success (drone ship)      14
Success (ground pad)       9
Controlled (ocean)         5
Failure (drone ship)       5
Failure                    3
Failure (parachute)        2
Uncontrolled (ocean)       2
Precluded (drone ship)     1
Name: count, dtype: int64

In [4]:
# Class label: success outcomes -> 1, everything else (failure, no attempt, controlled ocean ...) -> 0
df["Class"]=df.Landing_Outcome.str.startswith("Success").astype(int)
df[["Landing_Outcome","Class"]].drop_duplicates().sort_values("Class")

,Landing_Outcome,Class
0,Failure (parachute),0
2,No attempt,0
5,Uncontrolled (ocean),0
8,Controlled (ocean),0
13,Failure (drone ship),0
18,Precluded (drone ship),0
64,Failure,0
19,Success (ground pad),1
22,Success (drone ship),1
57,Success,1


In [5]:
print("Overall landing success rate:",round(df.Class.mean(),3))
df.groupby(df.Date.str[:4])["Class"].mean().round(3)

Overall landing success rate: 0.604


Date
2010    0.000
2012    0.000
2013    0.000
2014    0.000
2015    0.143
2016    0.625
2017    0.778
2018    0.500
2019    0.909
2020    0.875
Name: Class, dtype: float64

In [6]:
df["FlightNumber"]=np.arange(1,len(df)+1)
df.to_csv("data/spacex_wrangled.csv",index=False); df.head()

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome,Class,FlightNumber
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute),0,1
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute),0,2
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt,0,3
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt,0,4
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt,0,5
